# RAG Faithfulness Study — Interactive Analysis

This notebook lets you explore the experimental data interactively.  
Run the pipeline first (`python scripts/run_experiment.py`) to populate `results/`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import json
from pathlib import Path

%matplotlib inline
matplotlib.rcParams['figure.dpi'] = 120

ROOT = Path('..')
DATA = ROOT / 'data'
RES  = ROOT / 'results'

print('Paths ready')

## 1. Load Data

In [ ]:
questions   = pd.read_csv(DATA / 'questions.csv')
passages    = pd.read_csv(DATA / 'passages.csv')
baseline    = pd.read_csv(DATA / 'baseline_answers.csv')
rag         = pd.read_csv(DATA / 'rag_answers.csv')
scores      = pd.read_csv(DATA / 'evaluation_scores.csv')

print('Questions:', len(questions))
print('Passages:', len(passages))
print('Scores rows:', len(scores))
scores.head()

## 2. Summary Statistics

In [ ]:
summary = scores.groupby('answer_type')[['correctness','faithfulness','completeness','hallucination_rate']].mean().round(3)
print(summary)
summary

In [ ]:
# Total hallucinations
scores.groupby('answer_type')['hallucination_count'].sum()

## 3. Visualisations

In [ ]:
# Grouped bar chart for main metrics
metrics = ['correctness', 'faithfulness', 'completeness']
b = scores[scores.answer_type=='baseline'][metrics].mean()
r = scores[scores.answer_type=='rag'][metrics].mean()

x  = range(len(metrics))
w  = 0.35
fig, ax = plt.subplots(figsize=(8,5))
ax.bar([xi - w/2 for xi in x], b, w, label='Baseline', color='#E07B54')
ax.bar([xi + w/2 for xi in x], r, w, label='RAG',      color='#3A7CA5')
ax.set_xticks(list(x))
ax.set_xticklabels([m.capitalize() for m in metrics])
ax.set_ylim(0,6); ax.set_ylabel('Score (1-5)')
ax.set_title('Average Quality Metrics: Baseline vs RAG')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Per-question faithfulness
pivot = scores.pivot(index='question_id', columns='answer_type', values='faithfulness')
pivot.plot(kind='bar', figsize=(10,4), color=['#E07B54','#3A7CA5'])
plt.title('Per-Question Faithfulness')
plt.ylabel('Score (1-5)')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Condition')
plt.tight_layout()
plt.show()

In [ ]:
# Hallucination comparison
hall = scores.groupby('answer_type')['hallucination_rate'].mean()
hall.plot(kind='bar', color=['#E07B54','#3A7CA5'], figsize=(5,4))
plt.title('Average Hallucination Rate')
plt.ylabel('Rate')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Category Analysis

In [ ]:
cat_summary = scores.groupby(['category','answer_type'])['faithfulness'].mean().unstack()
cat_summary.plot(kind='bar', figsize=(9,4), color=['#E07B54','#3A7CA5'])
plt.title('Faithfulness by Category')
plt.ylabel('Avg Faithfulness (1-5)')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Condition')
plt.tight_layout()
plt.show()

## 5. Browse Individual Answers

In [ ]:
# Change question_id to any Q01–Q10
QID = 'Q03'

q_text = questions[questions.id == QID]['question'].values[0]
passage = passages[passages.question_id == QID]['passage'].values[0]
b_ans = baseline[baseline.question_id == QID]['answer'].values[0]
r_ans = rag[rag.question_id == QID]['answer'].values[0]
row_b = scores[(scores.question_id==QID) & (scores.answer_type=='baseline')].iloc[0]
row_r = scores[(scores.question_id==QID) & (scores.answer_type=='rag')].iloc[0]

print(f'QUESTION ({QID}): {q_text}\n')
print('─'*70)
print('REFERENCE PASSAGE:')
print(passage)
print('\n' + '─'*70)
print(f'BASELINE ANSWER  [Correct={row_b.correctness} Faith={row_b.faithfulness} Halluc={row_b.hallucination_count}]')
print(b_ans)
print('\n' + '─'*70)
print(f'RAG ANSWER       [Correct={row_r.correctness} Faith={row_r.faithfulness} Halluc={row_r.hallucination_count}]')
print(r_ans)

## 6. Correlation Analysis

In [ ]:
num_cols = ['correctness','faithfulness','completeness','hallucination_rate']
corr = scores[num_cols].corr().round(2)
print('Correlation matrix:')
corr